In [1]:
import pandas as pd
import mysql.connector


# =========================
# 1. load csv file
# =========================

file_path = r"D:\major project 4-2\indian data sets without using api\electricity\Mumbai Electricity Consumption Dataset\mumbai_electricity_final_realistic.csv"

df = pd.read_csv(file_path)

# convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# replace NaN with None for MySQL
df = df.where(pd.notnull(df), None)

print("Dataset Loaded")
print("Total rows:", len(df))


# =========================
# 2. connect mysql (safe)
# =========================

conn = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="2711",
    database="smart_city",
    autocommit=False
)

cursor = conn.cursor()

# check database
cursor.execute("SELECT DATABASE();")
print("Connected DB:", cursor.fetchone())


# =========================
# 3. insert query
# =========================

insert_query = """
INSERT INTO electricity_mumbai (

timestamp,
zone,
household_kwh,
commercial_kwh,
industrial_kwh,
total_demand_kwh,
thermal_power_kwh,
solar_power_kwh,
hydro_power_kwh,
gas_power_kwh,
wind_power_kwh,
renewable_kwh,
distribution_loss_percent,
grid_load_percent,
power_outage,
temperature_c,
humidity_percent

)

VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)

"""


# =========================
# 4. prepare data list
# =========================

data = []

for _, row in df.iterrows():

    values = (

        row["timestamp"],
        row["zone"],
        row["household_kwh"],
        row["commercial_kwh"],
        row["industrial_kwh"],
        row["total_demand_kwh"],
        row["thermal_power_kwh"],
        row["solar_power_kwh"],
        row["hydro_power_kwh"],
        row["gas_power_kwh"],
        row["wind_power_kwh"],
        row["renewable_kwh"],
        row["distribution_loss_percent"],
        row["grid_load_percent"],
        row["power_outage"],
        row["temperature_c"],
        row["humidity_percent"]

    )

    data.append(values)


print("Rows to insert:", len(data))


# =========================
# 5. insert data
# =========================

try:

    cursor.executemany(insert_query, data)
    conn.commit()

    print("Data inserted successfully")

except Exception as e:

    print("Insert error:", e)


# =========================
# 6. check rows
# =========================

cursor.execute("SELECT COUNT(*) FROM electricity_mumbai")
print("Total rows in table:", cursor.fetchone())


cursor.close()
conn.close()

Dataset Loaded
Total rows: 87600
Connected DB: ('smart_city',)
Rows to insert: 87600
Data inserted successfully
Total rows in table: (87600,)
